# Imports

In [1]:
from pySVA.sva_core import *
from pySVA.sva_calc import *

/home/mgeraeds/.conda/envs/dfm_proc_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import dfm_tools as dfmt
import numpy as np
import xarray as xr

/home/mgeraeds/.conda/envs/dfm_proc_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data description

In [2]:
data_description = {
    "nfaces": "mesh2d_nFaces",
    "depth": "mesh2d_nLayers",
    "time" : "time"
}

# Set up dask cluster

In [3]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

In [131]:
n_cores = 4
n_processes = 2
mem_lim = str(int(np.floor(336/(n_cores*n_processes))))+'GB'

In [134]:
cluster = SLURMCluster(name='dask-cluster',
                       cores=n_cores,
                       memory=mem_lim,
                       processes=n_processes,
                       interface='ib0',
                       queue='genoa',
                       walltime='04:00:00',
                       asynchronous=0)

/home/mgeraeds/.conda/envs/dfm_proc_env/lib/python3.9/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 33837 instead
  warnings.warn(


In [135]:
print(cluster.job_script())

#!/usr/bin/env bash

#SBATCH -J dask-worker
#SBATCH -p genoa
#SBATCH -n 1
#SBATCH --cpus-per-task=4
#SBATCH --mem=40G
#SBATCH -t 04:00:00

/home/mgeraeds/.conda/envs/dfm_proc_env/bin/python -m distributed.cli.dask_worker tcp://172.22.63.191:43139 --nthreads 2 --nworkers 2 --memory-limit 19.56GiB --name dummy-name --nanny --death-timeout 60 --interface ib0



In [136]:
cluster.scale(1)

In [137]:
client = Client(cluster)

In [138]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://172.22.63.191:33837/status,
Dashboard: http://172.22.63.191:33837/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://172.22.63.191:43139,Workers: 0
Dashboard: http://172.22.63.191:33837/status,Total threads: 0
Started: Just now,Total memory: 0 B


# Load dataset

In [10]:
file_nc = r'/projects/0/einf1300/saltis-wp3-1/C_Work/04_ZUNO-zd_3d_2022/computations/B04/B04_2022_jun23-24/ZUNO-zd_dflowfm_B04_jun23-24_0000_map.nc'

In [14]:
data_xr = dfmt.open_partitioned_dataset(file_nc.replace('_0000_',"_0*_"), chunks={"time":1})#, "mesh2d_nFaces":5000, "mesh2d_nNodes":5000, "mesh2d_nEdges":5000})

>> xu.open_dataset() with 96 partition(s): 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 : 121.73 sec
>> xu.merge_partitions() with 96 partition(s): 3.68 sec
>> some variables dropped with merging of partitions: ['mesh2d_face_x_bnd', 'mesh2d_face_y_bnd']
>> dfmt.open_partitioned_dataset() total: 125.43 sec


In [15]:
uds = data_xr

In [16]:
import xugrid as xu

In [17]:
def build_face_edge_connectivity(uds):
    # First check if the provided dataset is a xu.core.wrap.UgridDataset

    if isinstance(uds, xu.core.wrap.UgridDataset):
        # > Get fill value, grid name, and dimensions
        fill_value = uds.grid.fill_value
        gridname = uds.grid.name
        dimn_faces = uds.grid.face_dimension
        dimn_maxfn = uds.grid.to_dataset().mesh2d.attrs['max_face_nodes_dimension']

        # > Get coordinate names
        coord_face_x, coord_face_y = uds.grid.to_dataset().mesh2d.attrs['node_coordinates'].replace('node', 'face').split()

        # > Get connectivity
        face_edges = uds.grid.face_edge_connectivity

        # > Make into xr.DataArray with correct sizes, dimensions, and coordinates
        face_edge_connectivity = xr.DataArray(face_edges, dims=[dimn_faces, dimn_maxfn],
                                              coords={f'{coord_face_x}': ([dimn_faces], uds[f'{coord_face_x}']),
                                                      f'{coord_face_y}': ([dimn_faces], uds[f'{coord_face_y}'])},
                                              attrs={'cf_role': 'face_edge_connectivity', 'start_index': 0,
                                                     '_FillValue': fill_value}, name=f'{gridname}_face_edges')

        
    else:
        raise IOError("Please provide xu.core.wrap.UgridDataset to be able to automatically derive connectivities of the unstructured grid.")

    return face_edge_connectivity


In [18]:
def build_edge_node_connectivity(uds):

    # > Get fill value, grid name and dimensions 
    fill_value = uds.grid.fill_value
    gridname = uds.grid.name
    dimn_edges = uds.grid.edge_dimension

    # > Get coordinate names
    coord_edge_x, coord_edge_y = uds.grid.to_dataset().mesh2d.attrs['node_coordinates'].replace('node', 'edge').split()

    # > Determine dimension name
    dimn_maxen = f'{gridname}_nMax_edge_nodes'

    # > Get connectivity
    edge_nodes = uds.grid.edge_node_connectivity

    # > Make into xr.DataArray with correct sizes, dimensions, and coordinates
    edge_node_connectivity = xr.DataArray(data=edge_nodes, dims=[dimn_edges, dimn_maxen], coords={f'{coord_edge_x}':([dimn_edges], uds[f'{coord_edge_x}']), f'{coord_edge_y}':([dimn_edges], uds[f'{coord_edge_y}'])}, attrs={'cf_role': 'edge_node_connectivity', 'start_index':0, '_FillValue':fill_value}, name=uds.grid.to_dataset().mesh2d.attrs['edge_node_connectivity'])

    return edge_node_connectivity

In [19]:

def get_all_coordinates(uds):

    # > Get coordinate names
    coord_face_x, coord_face_y = uds.grid.to_dataset().mesh2d.attrs['node_coordinates'].replace('node', 'face').split()
    coord_edge_x, coord_edge_y = uds.grid.to_dataset().mesh2d.attrs['node_coordinates'].replace('node', 'edge').split()
    coord_node_x, coord_node_y = uds.grid.to_dataset().mesh2d.attrs['node_coordinates'].split()

    # > Get dimension names
    dimn_faces = uds.grid.face_dimension
    dimn_nodes = uds.grid.node_dimension
    dimn_edges = uds.grid.edge_dimension

    # > Get grid name
    gridname = uds.grid.name

    # > Get face coordinates
    face_array = np.c_[uds.mesh2d_face_x, uds.mesh2d_face_y] # is NOT equal to uds.grid.face_coordinates
    face_coords = xr.DataArray(data=face_array, dims=[dimn_faces,f'{gridname}_nCartesian_coords'], coords={f'{coord_face_x}':([dimn_faces], uds[f'{coord_face_x}']), f'{coord_face_y}':([dimn_faces], uds[f'{coord_face_y}'])}, attrs={'units':'m', 'standard_name': 'projection_x_coordinate, projection_y_coordinate', 'long_name':'Characteristic coordinates of mesh face', 'bounds': 'mesh2d_face_x_bnd, mesh_face_y_bnd'})                                                                                                            

    # > Get edge coordaintes
    edge_array = uds.grid.edge_coordinates # np.c_[uds.mesh2d_edge_x, uds.mesh2d_edge_y]
    edge_coords = xr.DataArray(data=edge_array, dims=[dimn_edges,f'{gridname}_nCartesian_coords'], coords={f'{coord_edge_x}':([dimn_edges], uds[f'{coord_edge_x}']), f'{coord_edge_y}':([dimn_edges], uds[f'{coord_edge_y}'])}, attrs={'units':'m', 'standard_name': 'projection_x_coordinate, projection_y_coordinate', 'long_name':'Characteristic coordinates of mesh face', 'bounds': 'mesh2d_face_x_bnd, mesh_face_y_bnd'})   

    # > Get node coordinates
    node_array =  uds.grid.node_coordinates # np.c_[uds.mesh2d_node_x, uds.mesh2d_node_y]
    node_coords = xr.DataArray(data=node_array, dims=[dimn_nodes,f'{gridname}_nCartesian_coords'], coords={f'{coord_node_x}':([dimn_nodes], uds[f'{coord_node_x}']), f'{coord_node_y}':([dimn_nodes], uds[f'{coord_node_y}'])}, attrs={'units':'m', 'standard_name': 'projection_x_coordinate, projection_y_coordinate', 'long_name':'Characteristic coordinates of mesh node', 'bounds': 'mesh2d_node_x_bnd, mesh_node_y_bnd'})   

    return face_coords, edge_coords, node_coords

In [20]:
def calculate_unit_normal_vectors(uds, **kwargs):

    # > Get dimensions, gridname, and coordinates
    dimn_edges = uds.grid.edge_dimension
    dimn_nodes = uds.grid.node_dimension
    fill_value = uds.grid.fill_value
    gridname = uds.grid.name

    varname_unvs = f'{gridname}_unvs'

    # > See if flow area is already in the variables
    if varname_unvs in uds.variables:
        print(f'Unit normal vectors on edges already present Dataset in variable {varname_unvs}.')
        return uds[varname_unvs]

    else:
        # > Get edge coordinate names
        coord_edge_x, coord_edge_y = uds.grid.to_dataset().mesh2d.attrs['node_coordinates'].replace('node', 'edge').split()

        # > Get kwargs
        edge_node_coords = kwargs.get('edge_node_coords')
        # face_edges = kwargs.get('face_edges')

        # > See if node-edge connectivity is given as a kwarg. If not, reconstruct it
        if 'edge_node_coords' in kwargs:
            pass
        else:
            # > Get the node-edge connectivity
            edge_nodes = build_edge_node_connectivity(uds)

            # > Build the node_coords
            _, _, node_coords = get_all_coordinates(uds) # just get the node_coords

            # > Get the coordinates of all nodes belonging to an edge
            edge_node_coords = xr.where(edge_nodes!=fill_value, node_coords.isel({dimn_nodes:edge_nodes}), np.nan)
        
        x1 = edge_node_coords[:,0,0]
        x2 = edge_node_coords[:,1,0]
        y1 = edge_node_coords[:,0,1]
        y2 = edge_node_coords[:,1,1]
        x = x2 - x1
        y = y2 - y1
        
        nf = np.dstack([-y,x])
        
        # > Calculate the norm and divide by the norm
        unv = nf / np.linalg.norm(nf)
        edge_unvs = unv[0]
        
        # > Put it in xr.DataArray format
        edge_unvs = xr.DataArray(data=edge_unvs, dims=[dimn_edges,f'{gridname}_nCartesian_coords'], coords={f'{coord_edge_x}':([dimn_edges], uds[f'{coord_edge_x}']), f'{coord_edge_y}':([dimn_edges], uds[f'{coord_edge_y}'])})  
        uds[f'{varname_unvs}'] = edge_unvs
        
        return edge_unvs

In [21]:
dimn_maxfn = uds.ugrid.grid.to_dataset().mesh2d.attrs['max_face_nodes_dimension']
dimn_faces = uds.ugrid.grid.face_dimension
dimn_edges = uds.ugrid.grid.edge_dimension
fill_value = uds.ugrid.grid.fill_value
gridname = uds.ugrid.grid.name
dimn_maxef = f'{gridname}_nMax_edge_faces'

In [22]:
from pySVA.sva_helpers import calculate_distance_haversine, calculate_distance_pythagoras

In [23]:
def build_edge_face_weights(uds):

	# Get dimension names
    dimn_maxfn = uds.ugrid.grid.to_dataset().mesh2d.attrs['max_face_nodes_dimension']
    dimn_faces = uds.ugrid.grid.face_dimension
    dimn_edges = uds.ugrid.grid.edge_dimension
    fill_value = uds.ugrid.grid.fill_value
    gridname = uds.ugrid.grid.name
    dimn_maxef = f'{gridname}_nMax_edge_faces'
	
    # > Get the edge-face connectivity
    edge_faces = xr.DataArray(uds.ugrid.grid.edge_face_connectivity, dims=(dimn_edges, dimn_maxef))
    
    # > Get all relevant coordinates
    face_coords, edge_coords, _ = get_all_coordinates(uds)

	# > Fill edge-face-connectivity matrix with face coordinates
    edge_face_coords = xr.where(edge_faces!=fill_value, face_coords.isel({dimn_faces:edge_faces}), np.nan)

	# > Get variables for d1 (distance between neighbouring cell faces through edge)
	# > Obtain these from the edge_face_coords dataset
    x0 = edge_face_coords.isel({f'{gridname}_nMax_edge_faces':0, f'{gridname}_nCartesian_coords':0})
    x1 = edge_face_coords.isel({f'{gridname}_nMax_edge_faces':1, f'{gridname}_nCartesian_coords':0})
    y0 = edge_face_coords.isel({f'{gridname}_nMax_edge_faces':0, f'{gridname}_nCartesian_coords':1})
    y1 = edge_face_coords.isel({f'{gridname}_nMax_edge_faces':1, f'{gridname}_nCartesian_coords':1})
    
    d1 = calculate_distance_pythagoras(x0, y0, x1, y1)
    
    # > Then get variables for d2 (distance from cell face in the first column to edge)
    x2 = edge_coords.isel({f'{gridname}_nCartesian_coords':0})
    y2 = edge_coords.isel({f'{gridname}_nCartesian_coords':1})
    d2 = calculate_distance_pythagoras(x0, y0, x2, y2)
    
    # > Calculate the weights per edge:
    w = d2 / d1
    
    return w

In [24]:
unvs = calculate_unit_normal_vectors(uds)

In [25]:
edge_faces = xr.DataArray(uds.ugrid.grid.edge_face_connectivity, dims=(dimn_edges, dimn_maxef))

# > Get the face-edge connectivity
face_edges = build_face_edge_connectivity(uds)

# > Get the unit normal vectors (nf) also in the face-edges matrix
fe_nfs = xr.where(face_edges!=fill_value, unvs.isel({dimn_edges:face_edges}), np.nan)

In [41]:
varname = 'mesh2d_u1'
uda = uds[f'{varname}']

In [59]:
varname = 'mesh2d_sa1'
uda = uds[f'{varname}']

In [43]:
dimn_maxfn

'mesh2d_nMax_face_nodes'

In [45]:
# > Get the edge-face connectivity and replace fill values with -1
edge_faces = xr.DataArray(uds.ugrid.grid.edge_face_connectivity, dims=(dimn_edges, dimn_maxef))
edge_faces_validbool = edge_faces!=fill_value
edge_faces = edge_faces.where(edge_faces_validbool, -1)

# > Get the face-edge connectivity and replace fill values with -1
face_edges = xr.DataArray(grid.face_edge_connectivity, dims=(dimn_faces, dimn_maxfn))
face_edges_validbool = face_edges!=fill_value
face_edges = face_edges.where(face_edges_validbool, -1)

In [78]:
edge_var_part = edge_var.isel(time=4)

In [79]:
# > Calculate the variable on the edges, based on the face_weights
edge_var = face_weights * edge_var_part.isel({f'{dimn_maxef}': 0}) + (1 - face_weights) * edge_var_part.isel({f'{dimn_maxef}': 1})

In [38]:
face_weights = build_edge_face_weights(uds)

In [109]:
from dfm_tools.xugrid_helpers import get_vertical_dimensions

In [110]:
dimn_layer, dimn_interfaces = get_vertical_dimensions(uds)

In [ ]:

def compute_gradient(constructorSVA, varname, **kwargs):

    from dfm_tools.xugrid_helpers import get_vertical_dimensions

    # > Obtain uds from constructorSVA object
    uds = constructorSVA.ds

    # > Get grid
    grid = uds.grid

    # > Get dimension and grid names
    dimn_maxfn = grid.to_dataset().mesh2d.attrs['max_face_nodes_dimension']
    dimn_faces = grid.face_dimension
    dimn_edges = grid.edge_dimension
    fill_value = grid.fill_value
    gridname = grid.name
    dimn_maxef = f'{gridname}_nMax_edge_faces'
    dimn_layer, dimn_interfaces = get_vertical_dimensions(uds)

    # > Check kwargs
    if 'varname_unvs' in kwargs:
        varname_unvs = kwargs['varname_unvs'] 
    else:
        varname_unvs = f'{gridname}_unvs'

    # > Calculate the unit normal vectors if not in the dataset already
    try:
        unvs = uds[varname_unvs]
    except:
        # > And if not in the kwargs
        try:
            unvs = kwargs['unvs']
        except:
            unvs = calculate_unit_normal_vectors(uds)
        
    # > Get the edge-face connectivity and replace fill values with -1
    edge_faces = xr.DataArray(uds.ugrid.grid.edge_face_connectivity, dims=(dimn_edges, dimn_maxef))
    edge_faces_validbool = edge_faces!=fill_value
    edge_faces = edge_faces.where(edge_faces_validbool, -1)

    # > Get the face-edge connectivity and replace fill values with -1
    face_edges = build_face_edge_connectivity(uds)
    face_edges_validbool = face_edges!=fill_value
    face_edges = face_edges.where(face_edges_validbool, -1)

    # > Get the unit normal vectors (nf) also in the face-edges matrix
    fe_nfs = xr.where(face_edges!=fill_value, unvs.isel({dimn_edges:face_edges}), np.nan)

    # > Select only data-array of the to-be-used variable
    uda = uds[f'{varname}']

	# > Determine if we're looking at a velocity value u1 or u0
    # > Because these are vector quantities in the direction of the normal vector,
    # > to get to the final vector, we have to multiply u1/u0 by the normal vector
    # > first. Also, we need to check their sign for every edge.
    if varname == f'{gridname}_u1' or f'{gridname}_u0':

        # > Make sure the edge dimension is not chunked, otherwise we will 
        # > get "PerformanceWarning: Slicing with an out-of-order index is generating x times more chunks."
        chunks = {dimn_edges:-1}
        uda = uda.chunk(chunks)

        # > Fill the face-edges matrix with the varname
        # > Do this via stack and unstack since 2D indexing does not
        # > properly work in dask yet: https://github.com/dask/dask/pull/10237
        face_edges_stacked = face_edges.stack(__tmp_dim__=(dimn_faces, dimn_maxfn))
        edge_var_stacked = uda.isel({dimn_edges: face_edges_stacked})
        edge_var = edge_var_stacked.unstack("__tmp_dim__")
        # > Convert data-array back to an xu.UgridDataArray
        edge_var = xu.UgridDataArray(edge_var, grid=grid)

		# >> We have to determine the sign of the velocity 
		# >> Determine whether the u1 value is positive or negative
		# > Get the mesh2d_nFaces numbering of the 0th column in edge_faces (from
        # >  0 -> 1 is positive)
        pos_fe = xr.where(face_edges!=fill_value, edge_faces.isel({dimn_maxef:0}).isel({dimn_edges:face_edges}), fill_value)

		# > If the number of the 0th column in edge_faces == mesh2d_nFaces, then the 
        # > direction is already positive in the right direction.
		# > Otherwise, the direction needs to be flipped
        fe_multiplier = xr.where(pos_fe==uds[dimn_faces], 1, -1)
        edge_var = edge_var * fe_multiplier

        # > Multiply by the unit normal vector to get to a vector quantity 
        # > With the multiplication we intend to calculate the dot product
        edge_var = edge_var * fe_nfs

	# > Fill face_edge matrix with flow area data
    edge_au = uds[f'{gridname}_au'].isel({dimn_edges:face_edges})
	
	# > Multiply the variable with the edge area (flow area), multiply by the 
    # > "flipped boolean" and the unit normal vector, and sum (dimension: faces)
    face_vars = (edge_var * edge_au * fe_nfs).sum(dim=dimn_maxfn, keep_attrs=True)
	
	# > Multiply the total result with (1/cell volume) (dimension: faces)
    gradient = (1/uds[f'{gridname}_vol1']) * face_vars
    uds[f'{varname}_div'] = gradient
    
    return gradient


In [ ]:
def uda_to_edges(uda, **kwargs):

    # > Get grid
    grid = uda.grid

    # > Get dimension and grid names
    dimn_maxfn = grid.to_dataset().mesh2d.attrs['max_face_nodes_dimension']
    dimn_faces = grid.face_dimension
    dimn_edges = grid.edge_dimension
    fill_value = grid.fill_value
    gridname = grid.name
    dimn_maxef = f'{gridname}_nMax_edge_faces'

    # > Calculate the weights for the scalar interpolation 
    # > first, if not given in the kwargs.
    try: 
        face_weights = kwargs[face_weights]
    except:
        face_weights = build_edge_face_weights(uds)
    
    # > Make sure the face dimension is not chunked, otherwise we will 
    # > get "PerformanceWarning: Slicing with an out-of-order index is generating x times more chunks."
    chunks = {dimn_faces:-1}
    uda = uda.chunk(chunks)

    # > Select the varname on faces in the edge-face connectivity matrix
    edge_faces_stacked = edge_faces.stack(__tmp_dim__=(dimn_edges, dimn_maxef))
    edge_var_stacked = uda.isel({dimn_faces: edge_faces_stacked})
    edge_var = edge_var_stacked.unstack("__tmp_dim__")
    # > Convert data-array back to an xu.UgridDataArray
    edge_var = xu.UgridDataArray(edge_var, grid=grid)

    # > Set fill values to nan-values
    edge_var = edge_var.where(edge_faces_validbool, np.nan)

    # > Make sure the edge dimension is not chunked, otherwise we will 
    # > get "PerformanceWarning: Slicing with an out-of-order index is generating x times more chunks."
    if 'chunks' in kwargs:
        chunks = kwargs['chunks']
        edge_var = edge_var.chunk(chunks)
        face_weights = face_weights.chunk(chunks)
    else:
        pass        

    # > Calculate the variable on the edges, based on the face_weights
    edge_var = face_weights * edge_var.isel({f'{dimn_maxef}': 0}) + (1 - face_weights) * edge_var.isel({f'{dimn_maxef}': 1})

    # > Get the variables in the face-edges matrix for later multiplication
    # > with the flow area for the Green-Gauss theorem
    face_edges_stacked = face_edges.stack(__tmp_dim__=(dimn_faces, dimn_maxfn))
    edge_var_stacked = uda.isel({dimn_edges: face_edges_stacked})
    edge_var = edge_var_stacked.unstack("__tmp_dim__")
    # > Convert data-array back to an xu.UgridDataArray
    edge_var = xu.UgridDataArray(edge_var, grid=grid)

    

In [118]:
if 'time' in uda.dims:
    chunks = {'time':50, dimn_edges:50000, dimn_layer:5, 'mesh2d_nMax_edge_faces':1}
edge_var = edge_var.chunk(chunks)

In [139]:
# > Calculate the variable on the edges, based on the face_weights
ev = face_weights * edge_var.isel({f'{dimn_maxef}': 0}) + (1 - face_weights) * edge_var.isel({f'{dimn_maxef}': 1})

2024-02-28 22:34:01,567 - distributed.protocol.core - CRITICAL - Failed to Serialize
Traceback (most recent call last):
  File "/home/mgeraeds/.conda/envs/dfm_proc_env/lib/python3.9/site-packages/distributed/protocol/core.py", line 109, in dumps
    frames[0] = msgpack.dumps(msg, default=_encode_default, use_bin_type=True)
  File "/home/mgeraeds/.conda/envs/dfm_proc_env/lib/python3.9/site-packages/msgpack/__init__.py", line 38, in packb
    return Packer(**kwargs).pack(o)
  File "msgpack/_packer.pyx", line 294, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 300, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 297, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 264, in msgpack._cmsgpack.Packer._pack
  File "msgpack/_packer.pyx", line 231, in msgpack._cmsgpack.Packer._pack
  File "msgpack/_packer.pyx", line 264, in msgpack._cmsgpack.Packer._pack
  File "msgpack/_packer.pyx", line 272, in msgpack._cmsgpack.Packer._pack
ValueErr

CancelledError: ('all-aggregate-ba1dffc0d1cca0407f6d944b96725732',)

In [141]:
import dask

In [91]:
edge_vars = edge_var.where(edge_faces_validbool, np.nan)

In [142]:
dask.__version__

'2023.7.1'

In [132]:
uda = uds['mesh2d_sa1']

In [19]:
sva = constructorSVA(input_file=data_xr,
                     data_description=data_description)

sva.velu0 = data_xr.mesh2d_u0
sva.velu1 = data_xr.mesh2d_u1
# sva.velz = data_xr.mesh2d_ucz
sva.tracer = data_xr.mesh2d_sa1

# tracer_variance = compute_variance(tef)


In [20]:
sva